# RetroTide native designPKS adapter
Calls the RetroTide design API directly. Shares upstream package with BioPKS; does not double-count reactions.


In [ ]:
from pathlib import Path
import json,subprocess,zipfile,shutil
ROOT=Path('/content/synbiocrow_m2412');ROOT.mkdir(parents=True,exist_ok=True)

# Self-healing isolated BioPKS environment. No manual uploads or prior cells required.
import subprocess, shutil, urllib.request, time, os, json
from pathlib import Path
ROOT_ENV=Path('/content/biosynthesis_ensemble/engines/biopks_retrotide')
ENV=ROOT_ENV/'env'; PY=ENV/'bin/python'; REPO=ROOT_ENV/'source'; LOG=ROOT/'install.log'
ROOT_ENV.mkdir(parents=True,exist_ok=True)
PIN='372cbfe9caf452fdf83976e6d0f32511dd28fccb'
def cmd(args,timeout=2400):
 print('RUN', ' '.join(map(str,args)),flush=True)
 with LOG.open('a') as f:
  p=subprocess.run(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,text=True,timeout=timeout)
 if p.returncode:
  raise RuntimeError(f'Command failed (exit {p.returncode}); see {LOG}')
 return p

def te_ok():
 if not PY.is_file():return False
 p=subprocess.run([str(PY),'-c','from biopks_pipeline.retrotide.bcs import TE; from rdkit import Chem; print("TE_OK")'],capture_output=True,text=True,timeout=120)
 print('Environment probe:',p.returncode,p.stdout.strip(),p.stderr[-500:])
 return p.returncode==0
if not te_ok():
 print('Isolated BioPKS environment missing/incomplete; provisioning automatically.')
 MM=Path('/content/micromamba')
 if not MM.is_file():
  archive=ROOT/'micromamba.tar.bz2'
  urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest',archive)
  cmd(['tar','-xjf',archive,'-C',str(ROOT),'bin/micromamba'])
  shutil.copy(ROOT/'bin/micromamba',MM)
 if not PY.is_file():cmd([str(MM),'create','-y','-p',str(ENV),'-c','conda-forge','python=3.10','pip','rdkit'],timeout=3600)
 if not (REPO/'.git').exists():cmd(['git','clone','https://github.com/JBEI/BioPKS-Pipeline.git',str(REPO)],timeout=900)
 cmd(['git','-C',str(REPO),'checkout','--detach',PIN],timeout=180)
 actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
 assert actual==PIN,('Wrong upstream commit',actual)
 original=(REPO/'requirements.txt').read_text()
 filtered=[line for line in original.splitlines() if not line.strip().lower().replace(' ','').startswith('umap==0.1.1')]
 if len(filtered)==len(original.splitlines()):raise RuntimeError('Expected umap==0.1.1 pin absent; review upstream dependency contract')
 compat=ROOT/'requirements_colab_compat.txt';compat.write_text('\n'.join(filtered)+'\n')
 cmd([str(PY),'-m','pip','install','--upgrade','pip','wheel'],timeout=900)
 cmd([str(PY),'-m','pip','install','setuptools==80.9.0'],timeout=900)
 cmd([str(PY),'-m','pip','install','-r',str(compat)],timeout=3600)
 cmd([str(PY),'-m','pip','install','--no-deps','--no-build-isolation','-e',str(REPO)],timeout=1200)
 cmd([str(PY),'-m','pip','install','setuptools==80.9.0'],timeout=900)
 if not te_ok():raise RuntimeError('Native BioPKS TE still unavailable after automatic installation; see install.log')
print('Native TE ready:',PY)


In [ ]:
from pathlib import Path
import subprocess,json
ROOT=Path('/content/synbiocrow_retrotide_native');ROOT.mkdir(parents=True,exist_ok=True)
RUNNER="#!/usr/bin/env python3\nimport json,sys,traceback\nfrom pathlib import Path\nfrom rdkit import Chem\nfrom biopks_pipeline.retrotide.retrotide import designPKS\n\ndef main(out):\n out=Path(out);out.mkdir(parents=True,exist_ok=True)\n targets=[('hexanoic_acid','CCCCCC(=O)O'),('3_hydroxypropionic_acid','O=C(O)CCO'),('butanal','CCCC=O')]\n records=[]; reports=[]\n for name,smiles in targets:\n  report={'target':name,'smiles':smiles,'native_design_count':0,'errors':[]}\n  try:\n   rounds=designPKS(Chem.MolFromSmiles(smiles),maxDesignsPerRound=5,similarity='atompairs')\n   for ridx,group in enumerate(rounds):\n    for didx,(cluster,score,bound) in enumerate(group):\n     records.append({'target':name,'target_smiles':smiles,'round':ridx+1,'design_index':didx,\n       'score':float(score),'native_bound_smiles':Chem.MolToSmiles(bound,isomericSmiles=True),\n       'native_cluster_repr':repr(cluster)[:2500], 'native_retrotide_designPKS_called':True,\n       'native_te_called':False,'exact_target_match':False,'balanced_reaction_certified':False})\n     report['native_design_count']+=1\n  except Exception: report['errors'].append(traceback.format_exc()[-5000:])\n  reports.append(report)\n (out/'native_design_release_records.json').write_text(json.dumps(records,indent=2))\n (out/'retrotide_design_reports.json').write_text(json.dumps(reports,indent=2))\n summary={'adapter':'RetroTide','api':'biopks_pipeline.retrotide.retrotide.designPKS',\n 'native_designs':len(records),'native_te_calls':0,'exact_target_matches':0,\n 'independent_api_invocation_verified':len(records)>0,\n 'distinct_algorithm_from_BioPKS':False,'reaction_exported':False,\n 'note':'Direct native RetroTide designPKS call; BioPKS remains the separate TE-release adapter. No duplicate reaction counting.'}\n (out/'retrotide_api_summary.json').write_text(json.dumps(summary,indent=2));print(json.dumps(summary,indent=2))\n if not records:raise RuntimeError('RetroTide designPKS generated no designs; see reports')\nif __name__=='__main__':main(sys.argv[1])\n"
(ROOT/'retrotide_design_runner.py').write_text(RUNNER)
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
p=subprocess.run([str(PY),str(ROOT/'retrotide_design_runner.py'),str(OUT)],capture_output=True,text=True,timeout=1200)
print('RETROTIDE_API_RUN',p.returncode,p.stdout[-3000:],p.stderr[-2000:])
if p.returncode:raise RuntimeError('RetroTide native designPKS failed; see output')
